# 03 - RQ1 - Incremental value of the EEG score beyond demographics

**RQ1a** (BrainLat): beyond age, sex and education. **RQ1b** (Greek): beyond age and sex (one-sided, OR > 1), with an
E-value for the unmeasured education. **RQ1c**: is the effect the same in both cohorts?

**Method.** Logistic regression demographics vs demographics + EEG score (logit of the out-of-fold P(AD), standardised
within the cohort; OR per 1 SD; profile-likelihood CI). The p-value of the likelihood-ratio statistic comes from a
parametric bootstrap of the whole LOSO under H0 (labels simulated from P(AD | demographics), LOSO re-run), because
the naive test with cross-validated scores can be liberal (Höfling & Tibshirani, 2008; Shang et al., 2025). Sequential
stopping (Besag & Clifford, 1991). Discrimination of demographic and combined models: leave-one-out logistic regression;
ΔAUC is descriptive.

**Outputs:** outputs/results/incremental.json, outputs/tables/null_bootstrap_*.json,
outputs/tables/auc_by_model.csv, outputs/predictions/baselines_*.csv.

**Time:** 30 min to 6 h (the null bootstrap re-runs the LOSO up to 199 times per cohort; it stops early when the result
is clearly not significant, so it takes longest when there is an effect). Every replicate is cached: if interrupted,
running again continues where it stopped.

In [ ]:
# Setup: make the eegad package importable (also when it was not installed with `pip install -e .`)
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=RuntimeWarning)   # MNE notices about head size and 'boundary' events
warnings.filterwarnings("ignore", category=FutureWarning)

_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "analysis_plan.json").exists()), None)
if _root is None:
    raise FileNotFoundError("Open this notebook from inside the repository (eeg-ad-demographics/notebooks).")
if str(_root / "src") not in sys.path:
    sys.path.insert(0, str(_root / "src"))

In [ ]:
import hashlib
import json

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import roc_auc_score

from eegad.config import load_context
from eegad.data import load_table
from eegad.reporting import fmt, fmt_ci, fmt_metrics, fmt_p, read_json, write_json

from eegad.models import loo_logistic, null_bootstrap
from eegad.stats import (auc_ci, delta_auc_boot, e_value, incremental_test, logit_fit, nested_lr, safe_logit,
                         standardize)

ctx = load_context("03")
log, out, plan = ctx.log, ctx.out, ctx.plan
NB, SEED = ctx.n_boot, ctx.seed
COV_BL, COV_GR = ctx.cov_brainlat, ctx.cov_greek
df_bl, df_gr = load_table(out / "derived" / "brainlat_epochs.csv"), load_table(out / "derived" / "greek_epochs.csv")


def with_score(subjects, name):
    pred = load_table(out / "predictions" / f"oof_{name}_RF_11.csv")
    return subjects.merge(pred[["subject_id", "p"]].rename(columns={"p": "p_eeg"}), on="subject_id", how="left")


subj_bl = with_score(load_table(out / "derived" / "brainlat_subjects.csv"), "BrainLat")
subj_gr = with_score(load_table(out / "derived" / "greek_subjects.csv"), "Greek")
EEG_LABEL, COMB_LABEL = "EEG score (RF, LOSO)", "Demographics + EEG score"
INC, BOOT, BASE, DELTA, AUC_TABLES = {}, {}, {}, {}, []


def baselines(s, covs, models):
    """Leave-one-out logistic models at the subject level; the EEG score enters as the logit of the OOF P(AD)."""
    t = s.dropna(subset=covs + ["p_eeg"]).reset_index(drop=True).copy()
    t["eeg_logit"] = safe_logit(t["p_eeg"])
    res = {label: loo_logistic(t, cols) for label, cols in models.items()}
    res[EEG_LABEL] = t[["subject_id", "y", "p_eeg"]].rename(columns={"p_eeg": "p"})
    res[COMB_LABEL] = loo_logistic(t, covs + ["eeg_logit"])
    return res


def auc_table(models, cohort):
    rows = []
    for name, d in models.items():
        a, lo, hi = auc_ci(d["y"], d["p"], NB, SEED)
        rows.append({"cohort": cohort, "model": name, "AUC": a, "lo": lo, "hi": hi, "n": len(d)})
    return pd.DataFrame(rows)


def show_incremental(r, one_sided=False):
    p_txt = f"p = {fmt_p(r['p'])}" + (f" (one-sided {fmt_p(r['p_one_sided'])})" if one_sided else "")
    print(f"{r['cohort']} (n = {r['n']}: {r['n_AD']} AD, {r['n_HC']} HC) — EEG score given {r['covariates']}: "
          f"LR chi2(1) = {r['LR']:.2f}, {p_txt}; OR per 1 SD = {fmt_ci(r['OR'], r['OR_lo'], r['OR_hi'], 2)} ({r['method']})")


def run_null_bootstrap(cohort, df_ep, subj, covs, name, one_sided):
    print(f"Null bootstrap ({cohort}): re-runs the LOSO up to {ctx.rt['null_B']} times under H0; each replicate is cached ...")
    bn = null_bootstrap(df_ep, subj, covs, name, ctx, one_sided=one_sided)
    write_json(out / "tables" / f"null_bootstrap_{cohort}.json", {k: v for k, v in bn.items() if k != "naive"})
    side = "one-sided" if one_sided else "two-sided"
    print(f"Null-bootstrap p ({side}) = {fmt_p(bn['p'])} | {bn['replicates']} replicates, {bn['exceedances']} >= observed"
          f"{' (sequential stop: p clearly above 0.05)' if bn['stopped_early'] else ''} | naive p "
          f"{fmt_p(bn['naive']['p_one_sided'] if one_sided else bn['naive']['p'])}")
    return bn


def inc_keys(prefix, r, one_sided=False):
    log.keys_from(prefix, {"or.value": r["OR"], "or.lo": r["OR_lo"], "or.hi": r["OR_hi"], "lr": r["LR"],
                           "p_analytic" if prefix in ("rq1a", "rq1b") else "p":
                           r["p_one_sided"] if one_sided else r["p"]})

## RQ1a - BrainLat: beyond age, sex and education

In [ ]:
INC["BrainLat"] = r = incremental_test(subj_bl, COV_BL, "p_eeg", "BrainLat")
show_incremental(r)
BOOT["BrainLat"] = b = run_null_bootstrap("BrainLat", df_bl, subj_bl, COV_BL, "BL_RF_11", one_sided=False)
v = log.verdict("RQ1a", "adds" if b["p"] < 0.05 else "not demonstrated")
print(f"Reading (plan): {v}")
log.record("RQ1a", "BrainLat: EEG score beyond age, sex and education",
           f"chi2(1) = {r['LR']:.2f}; p (null bootstrap) = {fmt_p(b['p'])} [{b['replicates']} replicates]; "
           f"naive p = {fmt_p(r['p'])}; OR/SD = {fmt_ci(r['OR'], r['OR_lo'], r['OR_hi'], 2)} (profile CI, "
           f"descriptive); {r['method']}; n = {r['n']}", "Results; Table 2", f"reading: {v}")
inc_keys("rq1a", r)
log.keys_from("rq1a.null_bootstrap", {"t_obs": b["T_obs"], "p": b["p"], "replicates": b["replicates"],
                                      "exceedances": b["exceedances"]})

INC["BrainLat + site"] = rs = incremental_test(subj_bl, COV_BL + ["site_cl"], "p_eeg", "BrainLat")
print(f"Check with site (outside the reading rule): OR/SD = {fmt_ci(rs['OR'], rs['OR_lo'], rs['OR_hi'], 2)}, "
      f"p = {fmt_p(rs['p'])} ({rs['method']}; n = {rs['n']})")
log.record("RQ1a", "BrainLat: check with site (age, sex, education, site)",
           f"OR/SD = {fmt_ci(rs['OR'], rs['OR_lo'], rs['OR_hi'], 2)}; p = {fmt_p(rs['p'])}; {rs['method']}",
           "Results (one sentence)", "check; outside the reading rule")
log.keys_from("rq1a.site_check", {"or.value": rs["OR"], "or.lo": rs["OR_lo"], "or.hi": rs["OR_hi"], "p": rs["p"]})

BASE["BrainLat"] = baselines(subj_bl, COV_BL, {"Age": ["Age"], "Age + sex": ["Age", "sex_fem"],
                                               "Age + sex + education": COV_BL})
AUC_TABLES.append(auc_table(BASE["BrainLat"], "BrainLat"))
display(AUC_TABLES[-1].round(3))
demo = BASE["BrainLat"]["Age + sex + education"]
for label, key, other in ((f"{COMB_LABEL} − demographics", "combined_minus_demographics", BASE["BrainLat"][COMB_LABEL]),
                          (f"{EEG_LABEL} − demographics", "eeg_minus_demographics", BASE["BrainLat"][EEG_LABEL])):
    mm = other.merge(demo[["subject_id", "p"]], on="subject_id", suffixes=("", "_d"))
    DELTA[("BrainLat", label)] = d = delta_auc_boot(mm["y"], mm["p"], mm["p_d"], NB, SEED)
    print(f"Delta AUC {label}: {fmt_ci(*d)} (descriptive)")
    log.record("RQ1a", f"BrainLat: delta AUC {label}", fmt_ci(*d), "Results", "descriptive, not tested")
    log.keys_from(f"delta_auc.brainlat.{key}", dict(zip(("value", "lo", "hi"), d)))
MODEL_KEY = {"Age": "age", "Age + sex": "age_sex", "Age + sex + education": "age_sex_education",
             EEG_LABEL: "eeg", COMB_LABEL: "demographics_plus_eeg"}
for _, row in AUC_TABLES[-1].iterrows():
    log.record("RQ1a", f"BrainLat: AUC {row['model']}", fmt_ci(row["AUC"], row["lo"], row["hi"]), "Table 2",
               "LOSO" if row["model"] == EEG_LABEL else "logistic, leave-one-out")
    if row["model"] != EEG_LABEL:
        log.keys_from(f"auc.brainlat.{MODEL_KEY[row['model']]}", {"value": row["AUC"], "lo": row["lo"], "hi": row["hi"]})

## RQ1b - Greek cohort: beyond age and sex (E-value for the unmeasured education)

In [ ]:
INC["Greek"] = r = incremental_test(subj_gr, COV_GR, "p_eeg", "Greek")
show_incremental(r, one_sided=True)
BOOT["Greek"] = b = run_null_bootstrap("Greek", df_gr, subj_gr, COV_GR, "GR_RF_11", one_sided=True)
v = log.verdict("RQ1b", "adds" if b["p"] < 0.05 else "not demonstrated")
E_PT, E_CI = e_value(r["OR"], r["OR_lo"], r["OR_hi"])
print(f"Reading (plan): {v}")
print(f"E-value: {fmt(E_PT, 2)} for the OR and {fmt(E_CI, 2)} for the CI limit (minimum RR of an unmeasured "
      "confounder, with both the diagnosis and the score, to explain the effect away)")

# Reference for reading the E-value: how much education changed the EEG effect in BrainLat, where it was measured
bl_cc = subj_bl.dropna(subset=COV_BL + ["p_eeg"]).reset_index(drop=True)
r_wo = incremental_test(bl_cc, ["Age", "sex_fem"], "p_eeg", "BrainLat")
Xe = np.column_stack([np.ones(len(bl_cc)), standardize(bl_cc[COV_BL].to_numpy(float))])
fe = logit_fit(Xe, bl_cc["y"].to_numpy(int))
OR_EDU = float(np.exp(fe["beta"][1 + COV_BL.index("years_education")]))
print(f"Reference (BrainLat, n = {len(bl_cc)}): EEG OR/SD = {r_wo['OR']:.2f} with age + sex and "
      f"{INC['BrainLat']['OR']:.2f} with + education (ratio {INC['BrainLat']['OR'] / r_wo['OR']:.2f}); "
      f"education: OR per SD of {OR_EDU:.2f} for AD (RR approx. {np.sqrt(OR_EDU if OR_EDU >= 1 else 1 / OR_EDU):.2f}), "
      "adjusted for age and sex")
log.record("RQ1b", "Reference for the E-value (BrainLat)",
           f"EEG OR/SD {r_wo['OR']:.2f} (age + sex) -> {INC['BrainLat']['OR']:.2f} (+ education); "
           f"education OR/SD {OR_EDU:.2f} for AD", "Discussion", "how much education changed the effect where measured")
log.keys_from("rq1b.reference", {"eeg_or_age_sex": r_wo["OR"], "eeg_or_age_sex_education": INC["BrainLat"]["OR"],
                                 "education_or": OR_EDU})

BASE["Greek"] = baselines(subj_gr, COV_GR, {"Age": ["Age"], "Age + sex": COV_GR})
AUC_TABLES.append(auc_table(BASE["Greek"], "Greek"))
display(AUC_TABLES[-1].round(3))
demo = BASE["Greek"]["Age + sex"]
for label, key, other in ((f"{COMB_LABEL} − demographics", "combined_minus_demographics", BASE["Greek"][COMB_LABEL]),
                          (f"{EEG_LABEL} − demographics", "eeg_minus_demographics", BASE["Greek"][EEG_LABEL])):
    mm = other.merge(demo[["subject_id", "p"]], on="subject_id", suffixes=("", "_d"))
    DELTA[("Greek", label)] = d = delta_auc_boot(mm["y"], mm["p"], mm["p_d"], NB, SEED)
    print(f"Delta AUC {label}: {fmt_ci(*d)} (descriptive)")
    log.record("RQ1b", f"Greek: delta AUC {label}", fmt_ci(*d), "Results", "descriptive, not tested")
    log.keys_from(f"delta_auc.greek.{key}", dict(zip(("value", "lo", "hi"), d)))
log.record("RQ1b", "Greek: EEG score beyond age and sex (one-sided)",
           f"chi2(1) = {r['LR']:.2f}; one-sided p (null bootstrap) = {fmt_p(b['p'])} [{b['replicates']} replicates]; "
           f"one-sided naive p = {fmt_p(r['p_one_sided'])}; OR/SD = {fmt_ci(r['OR'], r['OR_lo'], r['OR_hi'], 2)} "
           f"(profile CI, descriptive); {r['method']}; n = {r['n']}", "Results; Table 2", f"reading: {v}")
log.record("RQ1b", "Greek: E-value (education not recorded)", f"{fmt(E_PT, 2)} (OR); {fmt(E_CI, 2)} (CI limit)",
           "Results; Discussion")
inc_keys("rq1b", r, one_sided=True)
log.keys_from("rq1b.null_bootstrap", {"t_obs": b["T_obs"], "p": b["p"], "replicates": b["replicates"],
                                      "exceedances": b["exceedances"]})
log.keys_from("rq1b", {"e_value": E_PT, "e_value_ci": E_CI})
for _, row in AUC_TABLES[-1].iterrows():
    log.record("RQ1b", f"Greek: AUC {row['model']}", fmt_ci(row["AUC"], row["lo"], row["hi"]), "Table 2",
               "LOSO" if row["model"] == EEG_LABEL else "logistic, leave-one-out")
    if row["model"] != EEG_LABEL:
        log.keys_from(f"auc.greek.{MODEL_KEY[row['model']]}", {"value": row["AUC"], "lo": row["lo"], "hi": row["hi"]})

## RQ1c - Is the effect the same in both cohorts?

One-stage pooled model with cohort-specific intercept and covariates (BrainLat: age, sex, education; Greek: age, sex)
and a common standardised EEG score; heterogeneity: + cohort × EEG (LR, 1 df).

In [ ]:
def z_within(v):
    return standardize(np.asarray(v, float)[:, None])[:, 0]


bl_p = subj_bl.dropna(subset=COV_BL + ["p_eeg"]).reset_index(drop=True)
gr_p = subj_gr.dropna(subset=COV_GR + ["p_eeg"]).reset_index(drop=True)
n_b, n_g = len(bl_p), len(gr_p)
cohort = np.r_[np.zeros(n_b), np.ones(n_g)]
Zc = np.zeros((n_b + n_g, len(COV_BL) + len(COV_GR)))      # zeros = covariate of the other cohort
for j, c in enumerate(COV_BL):
    Zc[:n_b, j] = z_within(bl_p[c])
for j, c in enumerate(COV_GR):
    Zc[n_b:, len(COV_BL) + j] = z_within(gr_p[c])
z_eeg = np.r_[z_within(safe_logit(bl_p["p_eeg"])), z_within(safe_logit(gr_p["p_eeg"]))]
yv = np.r_[bl_p["y"].to_numpy(int), gr_p["y"].to_numpy(int)]
X0 = np.column_stack([np.ones(len(yv)), cohort, Zc])
X1 = np.column_stack([X0, z_eeg])
X2 = np.column_stack([X1, z_eeg * cohort])
R_POOL = nested_lr(X0, X1, yv, X1.shape[1] - 1)
R_HET = nested_lr(X1, X2, yv, X2.shape[1] - 1)
same_side = (INC["BrainLat"]["OR"] - 1) * (INC["Greek"]["OR"] - 1) > 0
v = log.verdict("RQ1c", "consistent" if (R_HET["p"] >= 0.05 and same_side) else "heterogeneous or uncertain")
print(f"Pooled (n = {len(yv)}: {n_b} BrainLat + {n_g} Greek): OR/SD = "
      f"{fmt_ci(R_POOL['OR'], R_POOL['OR_lo'], R_POOL['OR_hi'], 2)}, LR chi2(1) = {R_POOL['LR']:.2f}, "
      f"p = {fmt_p(R_POOL['p'])} ({R_POOL['method']})")
print(f"Cohort x EEG interaction: chi2(1) = {R_HET['LR']:.2f}, p = {fmt_p(R_HET['p'])} | ORs on the same side: {same_side}")
print(f"Reading (plan): {v}")
log.record("RQ1c", "Pooled: EEG score beyond the covariates of each cohort (common effect)",
           f"OR/SD = {fmt_ci(R_POOL['OR'], R_POOL['OR_lo'], R_POOL['OR_hi'], 2)}; chi2(1) = {R_POOL['LR']:.2f}, "
           f"p = {fmt_p(R_POOL['p'])}; {R_POOL['method']}; n = {len(yv)}", "Results; Fig. 2")
log.record("RQ1c", "Heterogeneity (cohort x EEG)", f"chi2(1) = {R_HET['LR']:.2f}, p = {fmt_p(R_HET['p'])}", "Results",
           f"reading: {v}")
log.keys_from("rq1c.pooled", {"or.value": R_POOL["OR"], "or.lo": R_POOL["OR_lo"], "or.hi": R_POOL["OR_hi"],
                              "lr": R_POOL["LR"], "p": R_POOL["p"]})
log.keys_from("rq1c.heterogeneity", {"lr": R_HET["LR"], "p": R_HET["p"]})
r1, r2 = log.get_verdict("RQ1a"), log.get_verdict("RQ1b")
v = log.verdict("RQ1", "adds in both cohorts" if r1 == r2 == "adds" else "adds only in BrainLat" if r1 == "adds"
                else "adds only in the Greek cohort" if r2 == "adds" else "not demonstrated in either cohort")
print(f"\nRQ1 overall: {v}")

# Saved for notebooks 06 and 07
write_json(out / "results" / "incremental.json",
           {"INC": INC, "R_POOL": R_POOL, "R_HET": R_HET, "E_value": [E_PT, E_CI],
            "BOOT": {k: {kk: vv for kk, vv in bb.items() if kk not in ("T", "naive")} for k, bb in BOOT.items()},
            "DELTA": {f"{c}|{label}": d for (c, label), d in DELTA.items()}})
pd.concat(AUC_TABLES).to_csv(out / "tables" / "auc_by_model.csv", index=False)
for cohort_name, models in BASE.items():
    pd.concat([d.assign(model=m) for m, d in models.items()]).to_csv(
        out / "predictions" / f"baselines_{cohort_name}.csv", index=False)